In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


# 1. Training and Test Data

In [3]:
import numpy as np

from sklearn.preprocessing import LabelEncoder

from sklearn.decomposition import PCA

# Loading processed data
# Using a relative path

data_path = "/content/drive/MyDrive/processed_data/"

X_train = np.load(data_path + "X_train.npy")

X_test = np.load(data_path + "X_test.npy")

y_train = np.load(data_path + "y_train.npy")

y_test = np.load(data_path + "y_test.npy")

# Flattening the 3D spectrogram

X_train = X_train.reshape(X_train.shape[0], -1)

X_test = X_test.reshape(X_test.shape[0], -1)

# Label encoding
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)

y_test_enc = le.transform(y_test)

# PCA Setup
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train)

X_test_pca = pca.transform(X_test)

# 2. Training Setup (Random Forest)

In [4]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import f1_score

# n_estimators=100 is a standard baseline
# n_jobs=-1 allows the model to use all CPU cores (highly recommended for RF)

rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Fitting model on training data by flattening X_train

rf_baseline.fit(X_train, y_train_enc)

# Setting predictions

y_pred_enc = rf_baseline.predict(X_test)

# Converting the labels to genre names for comparison
y_pred_labels = le.inverse_transform(y_pred_enc)

# Analysis measurement
f1 = f1_score(y_test, y_pred_labels, average='macro')

print(f"Baseline Random Forest F1 Score: {f1}")

Baseline Random Forest F1 Score: 0.48469061467930796


# 3. Training Setup (PCA)

In [2]:
from sklearn.metrics import f1_score

from sklearn.decomposition import PCA


# setting up the predictions and calculating the f1 score when using PCA
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train)

X_test_pca = pca.transform(X_test)

rf_pca = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf_pca.fit(X_train_pca, y_train_enc)

y_pred_pca_enc = rf_pca.predict(X_test_pca)

y_pred_pca_labels = le.inverse_transform(y_pred_pca_enc)

f1_pca = f1_score(y_test, y_pred_pca_labels, average='macro')


print(f"PCA Random Forest F1 Score: {f1_pca}")

NameError: name 'X_train' is not defined

# 4. Hyperparameter Tuning
- BayesSearchCV to optimize the Random Forest.
- Using max_depth and min_samples_split to manage overfitting.
- They control how complex each individual tree can get in the dataset.

In [ ]:
!pip install scikit-optimize

In [ ]:
from skopt import BayesSearchCV

from skopt.space import Integer, Categorical

# Reinitializing

rf = RandomForestClassifier(random_state=42)

# Setting the search space based on audio classification
search_space = {"n_estimators": Integer(100, 500),         # Number of trees

    "max_depth": Integer(10, 50),              # Controls overfitting limiting height

    "min_samples_split": Integer(2, 10),       # Minimum samples to split node

    "max_features": Categorical(['sqrt', 'log2']) # Number of features in each split

}

# Bayesian Search
bayes_search_rf = BayesSearchCV(estimator=rf, search_spaces=search_space, scoring="f1_macro", n_iter=10,     # 10 iterations for Random Forest to maintain a balance
    cv=3,          # 3-folds

    n_jobs=-1,

    random_state=42,

    verbose=1
)

# Finding the best parameters

bayes_search_rf.fit(X_train, y_train_enc)


print("Overall Best Parameters (Random Forest):")

print(bayes_search_rf.best_params_)


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Overall Best Parameters (Random Forest):
OrderedDict({'max_depth': 42, 'max_features': 'sqrt', 'min_samples_split': 6, 'n_estimators': 386})


# 5. Tuning with PCA

In [ ]:
# Running the search again on a PCA

rf_pca = RandomForestClassifier(random_state=42)

bayes_search_rf_pca = BayesSearchCV(estimator=rf_pca, search_spaces=search_space, scoring="f1_macro", n_iter = 10, cv=3, n_jobs=-1, random_state=42, verbose=1)

bayes_search_rf_pca.fit(X_train_pca, y_train_enc)

print("Overall Best Parameters (PCA):")

print(bayes_search_rf_pca.best_params_)


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Overall Best Parameters (PCA):
OrderedDict({'max_depth': 42, 'max_features': 'sqrt', 'min_samples_split': 7, 'n_estimators': 421})
